In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

In [2]:
np.random.seed(42)

In [3]:
df = pd.read_csv(r"D:\CSE(DataScience)\Customer_Churn_Prediction\Datasets\Telco-Customer-Churn-dataset.csv")

In [4]:
if "customerID" in df.columns:
    df.drop("customerID", axis=1, inplace=True)

In [5]:
# Fix TotalCharges (convert to numeric + impute missing)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

In [6]:
# Encode target
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

X = df.drop("Churn", axis=1)
y = df["Churn"]

In [7]:
# ----------------- Train/Test Split (Stratified) -----------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [8]:
# ----------------- Preprocessing -----------------
num_features = X.select_dtypes(include=["int64", "float64"]).columns
cat_features = X.select_dtypes(include=["object"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ]
)

In [10]:
# ----------------- Models -----------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=500, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel="rbf", probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

In [11]:
# ----------------- Train + Evaluate -----------------
results = []
MODEL_DIR = r"D:\CSE(DataScience)\Customer_Churn_Prediction\models"
os.makedirs(MODEL_DIR, exist_ok=True)

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append([name, acc, prec, rec, f1, auc])

In [13]:
# ---------- Save with clear consistent names ----------
filename_map = {
    "Logistic Regression": "churn_pipeline_logistic_regression.pkl",
    "KNN": "churn_pipeline_knn.pkl",
    "SVM": "churn_pipeline_svm.pkl",
    "Random Forest": "churn_pipeline_random_forest.pkl"
}

In [14]:
for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append([name, acc, prec, rec, f1, auc])

    # ✅ Save with consistent filename
    save_path = os.path.join(MODEL_DIR, filename_map[name])
    joblib.dump(pipeline, save_path)

    print(f"✅ Saved {name} pipeline at: {save_path}")

✅ Saved Logistic Regression pipeline at: D:\CSE(DataScience)\Customer_Churn_Prediction\models\churn_pipeline_logistic_regression.pkl
✅ Saved KNN pipeline at: D:\CSE(DataScience)\Customer_Churn_Prediction\models\churn_pipeline_knn.pkl
✅ Saved SVM pipeline at: D:\CSE(DataScience)\Customer_Churn_Prediction\models\churn_pipeline_svm.pkl
✅ Saved Random Forest pipeline at: D:\CSE(DataScience)\Customer_Churn_Prediction\models\churn_pipeline_random_forest.pkl
